# 01 · Data Cleaning and Data Quality

**Project:** Customer analytics and RFM segmentation on the UCI *Online Retail* dataset
**Scope of this notebook:** loading, auditing and cleaning the raw transactions, and saving a clean hand-off file for the next stage.

**Notebook map**
1. Business Understanding
2. Dataset Loading and Documentation
3. Data Audit
4. Cleaning and Data Quality Decisions
5. Feature Engineering
6. Hand-off and Summary

## 1. Business Understanding

The company is a UK-based online retailer selling gift items, many of them to wholesalers. This project turns raw transaction lines into customer behaviour (RFM) and then into actionable customer segments.

**Questions this notebook supports**
- Which parts of the data can we trust?
- How do guest checkouts, duplicates, returns and pricing errors distort revenue and customer counts?

**Working rule:** every cleaning decision is explained, quantified (rows affected) and logged with its business impact in section 4.5. Nothing is deleted silently.

## 2. Dataset Loading and Documentation

**Source:** UCI Machine Learning Repository, *Online Retail* — https://archive.ics.uci.edu/dataset/352/online+retail
**Content:** every transaction line of a UK-based online retailer between 1 Dec 2010 and 9 Dec 2011. One row is one product line on one invoice.

| Column | Meaning |
|---|---|
| `InvoiceNo` | Invoice number. A leading **C** marks a cancellation |
| `StockCode` | Product code. Some codes are not products (postage, fees, manual entries) |
| `Description` | Product name |
| `Quantity` | Units on the line. Negative for cancellations and returns |
| `InvoiceDate` | Date and time of the invoice |
| `UnitPrice` | Price per unit in pounds sterling |
| `CustomerID` | Customer identifier. Missing for guest or unlinked orders |
| `Country` | Customer's country |

`InvoiceNo` and `StockCode` are read as text on purpose: cancellations start with a letter and some stock codes contain letters.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA = Path("../data")
files = sorted(DATA.glob("online*retail.xlsx"))   # matches online_retail.xlsx and online-retail.xlsx
assert files, "Place the UCI Excel file in /data (see data/README.md)"

raw = pd.read_excel(files[0], dtype={"InvoiceNo": str, "StockCode": str})
print("File :", files[0].name)
print("Shape:", raw.shape)
raw.head()

## 3. Data Audit

We inspect the data before changing anything. `raw` stays untouched so every later step can be compared with it.

In [ ]:
df = raw.copy()

print("Dtypes:\n", df.dtypes, "\n")
print("Missing values:\n", df.isna().sum(), "\n")
print("Missing CustomerID: {:.1%} of rows".format(df["CustomerID"].isna().mean()))
print("Missing Description with UnitPrice = 0: {:.0%} of those rows".format(
    (df.loc[df["Description"].isna(), "UnitPrice"] == 0).mean()))
print("Exact duplicate rows:", df.duplicated().sum())
print("Date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("Unique invoices :", df["InvoiceNo"].nunique())
print("Unique customers:", df["CustomerID"].nunique())
print("Unique products :", df["StockCode"].nunique())
df.describe()

### Audit findings
- **Size and period:** 541,909 transaction lines and 8 columns, from 1 Dec 2010 to 9 Dec 2011. December 2011 is incomplete, so December must not be compared with December 2010.
- **Missing CustomerID:** 135,080 lines (about 25%). RFM needs an identifiable customer, so this is the main limit of the segmentation (investigated in 4.3).
- **Missing Description:** 1,454 lines. Not needed for RFM; product analysis relies on `StockCode`.
- **Duplicates:** 5,268 exact duplicate lines (about 1%), which would double-count revenue.
- **Anomalies:** `Quantity` (-80,995 to 80,995) and `UnitPrice` (-11,062 to 38,970) contain extreme and negative values (examined below).
- **Skew:** the median `Quantity` is 3 but the standard deviation is 218, so a few bulk or wholesale lines dominate the totals. This motivates the log transform before clustering.

### 3.1 Extreme quantities and prices

We list lines with an absolute quantity above 50,000 (using `abs()` catches both large sales and large returns), and the three highest and lowest unit prices.

In [ ]:
cols = ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]
print("Extreme quantities")
print(df.loc[df["Quantity"].abs() > 50000, cols].sort_values("InvoiceNo").to_string(), "\n")
print("Highest unit prices")
print(df.nlargest(3, "UnitPrice")[cols].to_string(), "\n")
print("Lowest unit prices")
print(df.nsmallest(3, "UnitPrice")[cols].to_string())

**Extreme quantities.** The four lines form two exact order-and-cancellation pairs: 74,215 units (invoice 541431, cancelled by C541433, customer 12346, £1.04) and 80,995 units (invoice 581483, cancelled by C581484, customer 16446, £2.08). Each pair has the same customer, product and price with opposite quantities, so it inflates gross volume but nets to zero revenue.

**Highest prices.** They belong to `Manual` and `AMAZONFEE` entries, which are charges and fees rather than product sales.

**Lowest prices.** They include negative prices labelled `Adjust bad debt` and zero-priced lines. These are accounting adjustments, not purchases.

**Consequence:** these entries must be handled before computing customer spending and order frequency, otherwise they distort Monetary and Frequency in RFM. They belong to the transactional cleaning stage (cancellations, zero and negative prices, non-product codes).

## 4. Cleaning and Data Quality Decisions

Order of operations: fix types, remove duplicates, investigate missing CustomerID, document special cases, then log every decision.

### 4.1 Fix data types
- `Description` becomes text. It mixes text with at least one numeric entry (see 4.4), which would otherwise break saving to Parquet.
- `InvoiceDate` becomes a true datetime so we can group by month, weekday and hour.
- `CustomerID` loads as a float (for example `17850.0`). We convert it to a text identifier so it can never be used as a numeric feature by mistake. The nullable integer type keeps missing values as `<NA>`.

In [ ]:
df["Description"] = df["Description"].astype("string").str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype("Int64").astype("string")

df[["Description", "InvoiceDate", "CustomerID"]].dtypes

### 4.2 Remove exact duplicates
A line that matches another line in every column, including the timestamp, is almost certainly a recording error. We remove duplicates before any counting so that invoice and customer counts are not inflated.

In [ ]:
n_before = len(df)
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_dup:,} duplicate rows ({n_before:,} -> {len(df):,})")

### 4.3 Investigate missing CustomerID
We do not drop these rows blindly. Grouping lines by invoice shows whether the missing IDs are scattered recording gaps (some lines of an invoice have an ID, others do not) or whole orders without a customer.

In [ ]:
df["HasCustomer"] = df["CustomerID"].notna()
rev = df["Quantity"] * df["UnitPrice"]            # line revenue, before returns are netted

per_invoice = df.groupby("InvoiceNo")["HasCustomer"].mean()
print("Invoices fully with customer:", (per_invoice == 1).sum())
print("Invoices fully guest        :", (per_invoice == 0).sum())
print("Invoices mixed              :", ((per_invoice > 0) & (per_invoice < 1)).sum())
print("Rows without CustomerID     : {:.1%}".format((~df["HasCustomer"]).mean()))
print("Revenue without CustomerID  : {:.1%}".format(rev[~df["HasCustomer"]].sum() / rev.sum()))

guest = df[~df["HasCustomer"]].copy()
df_known = df[df["HasCustomer"]].copy()
print("Guest rows:", len(guest), "| Known-customer rows:", len(df_known))

**Finding:** no invoice is partly missing its CustomerID. All 22,190 identified invoices carry an ID on every line, and all 3,710 other invoices carry none. The missing values are therefore whole guest or unlinked orders, not scattered recording gaps. They make up about 14% of invoices, 25.2% of lines and 14.9% of net revenue (before returns are netted out).

**Decision:** keep these rows for sales, product, time and geography analysis, where a customer ID is not needed. Exclude them from customer-level RFM and clustering, which require an identifiable customer.

**Limitation:** about 15% of revenue cannot be attributed to a customer, so the segmentation describes identified customers only.

### 4.4 Document the numeric Description entry
The check runs on `raw`, which still has the original types, so it can find values that are not text.

In [ ]:
odd = raw[raw["Description"].map(lambda x: not isinstance(x, str) and pd.notna(x))]
print("Non-text Description entries:", len(odd))
odd[["InvoiceNo", "StockCode", "Description", "UnitPrice"]]

This row has a stock code placed in the description field and a unit price of 0, which looks like a manual stock adjustment rather than a sale. It has no effect on revenue. Converting `Description` to text (4.1) removes the type conflict, and the zero-price row is handled with the other price anomalies in the transactional cleaning.

### 4.5 Data quality decisions log

In [ ]:
decisions = []
def log(issue, rows, decision, impact):
    decisions.append({"Issue": issue, "Rows": rows, "Decision": decision, "Business impact": impact})

log("Exact duplicate rows", n_dup, "Removed",
    "Prevents double-counting revenue, orders and customer activity")
log("Missing CustomerID", len(guest),
    "Whole invoices are guest orders (0 mixed). Kept for sales/product EDA, excluded from RFM",
    "About 15% of net revenue cannot be linked to a customer")
log("Missing Description", int(df["Description"].isna().sum()),
    "Kept as missing; product analysis uses StockCode",
    "None for RFM")
log("Non-text Description entry", len(odd),
    "Converted Description to text; zero-price row handled with price anomalies",
    "No revenue impact; avoids type errors when saving")
log("InvoiceDate and CustomerID types", len(df),
    "InvoiceDate to datetime; CustomerID to text identifier",
    "Enables time analysis; prevents numeric use of an ID")
log("Extreme quantities, negative and zero prices, cancellations", "not changed here",
    "Deferred to transactional cleaning", "Must be handled before computing Monetary and Frequency")

pd.DataFrame(decisions)

## 5. Feature Engineering

Calendar features for the EDA stage: seasonality by month, weekday and hour. `HasCustomer` was created in 4.3.

In [ ]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

df[["InvoiceDate", "Year", "Month", "DayOfWeek", "Hour", "HasCustomer"]].head()

## 6. Hand-off and Summary

We save the cleaned structural dataset. It is ignored by Git (see `.gitignore`), so teammates regenerate it by running this notebook.

In [ ]:
OUT = DATA / "interim_structural.parquet"
df.to_parquet(OUT, index=False)

check = pd.read_parquet(OUT)
assert check.shape == df.shape, "Saved file does not match the cleaned data"
print("Saved:", OUT.name, check.shape)
check.dtypes

### Summary
- 541,909 raw lines were reduced to 536,641 after removing 5,268 exact duplicates.
- Missing CustomerID occurs only on whole invoices (3,710 guest invoices, about 15% of net revenue). They stay available for sales analysis and are excluded from RFM.
- Types are fixed: `InvoiceDate` is a datetime, `CustomerID` is a text identifier and `Description` is text.
- The saved file `interim_structural.parquet` has 13 columns, including `Year`, `Month`, `DayOfWeek`, `Hour` and `HasCustomer`.

**Still to do in the next stage:** net out cancellations, isolate zero and negative prices, flag non-product stock codes, create `TotalLineRevenue` and `Is_UK`, and then build the customer-level RFM table.